# Round 2: RF tuning + feature engineering - S6E10

Experiments 4-7 (log: `experiments.md`). Same stratified 80/20 split, seed 42, so every number
compares directly with the log: logistic 0.92046, random forest 0.95351, LightGBM 0.95760.

This round follows Andrea's steer: tune the random forest, and test a proper feature set across ALL model families.
Discipline: one idea per experiment, failures logged, and no repeated tuning against this single holdout -
after this round the checkpoint is a 5-fold OOF re-score before any further parameter moves.


In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
LOG_REFERENCE = {'logistic': 0.92046, 'random_forest': 0.95351, 'lightgbm': 0.95760}
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']


In [2]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file() and Path('playground-series-s6e10/data/train.csv').is_file():
    competition_dir = Path('playground-series-s6e10')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')
test = pd.read_csv(data_dir / 'test.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])
test_features = test.drop(columns=['id'])

X_train, X_val, y_train, y_val = train_test_split(
    features, target, test_size=0.2, stratify=target, random_state=RANDOM_STATE
)

def to_lgbm(df):
    # LightGBM-native copy: categoricals as 'category', NaN left alone. sklearn frames keep object dtype.
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

print(f'Train: {len(X_train):,} / val: {len(X_val):,}')


Train: 559,708 / val: 139,927


## The engineered feature set (used in Exp 6 for every model)

Grounded in the importance tables (delays, service ratings, flight distance) rather than invented blindly:

- `total_delay`, `delay_diff` - carried over from Exp 3 for a fair retest inside a fuller set
- `dep_delayed`, `arr_delayed` - binary flags; being delayed at all may matter more than minutes
- `arr_delay_missing` - the 292 missing arrival delays are informative or noise; the flag lets the model decide
- `service_mean`, `service_min`, `service_std`, `low_service_count` (ratings <= 2) - overall impression and worst experience
- `log_flight_distance` - dampens the skewed distance distribution (helps the linear model most)


In [3]:
RATING_COLUMNS = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
    'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
    'Inflight entertainment', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Cleanliness',
]

def add_features(df):
    out = df.copy()
    dep = out['Departure Delay in Minutes']
    arr = out['Arrival Delay in Minutes']
    out['arr_delay_missing'] = arr.isna().astype(int)
    out['total_delay'] = dep + arr
    out['delay_diff'] = arr - dep
    out['dep_delayed'] = (dep > 0).astype(int)
    out['arr_delayed'] = (arr > 0).astype(int)
    out['service_mean'] = out[RATING_COLUMNS].mean(axis=1)
    out['service_min'] = out[RATING_COLUMNS].min(axis=1)
    out['service_std'] = out[RATING_COLUMNS].std(axis=1)
    out['low_service_count'] = (out[RATING_COLUMNS] <= 2).sum(axis=1)
    out['log_flight_distance'] = np.log1p(out['Flight Distance'])
    return out

X_train_fe = add_features(X_train)
X_val_fe = add_features(X_val)
print(f'{X_train.shape[1]} raw features -> {X_train_fe.shape[1]} with engineering')


21 raw features -> 31 with engineering


## Exp 4: random forest, first tuning move

Two deliberate changes from Exp 1, with reasons: `min_samples_leaf=5` smooths each leaf's estimate
(default 1 memorizes noise), and 400 trees instead of 200 (more trees reduce variance, but the benefit flattens and deep forests can still fit noise - tree count is no substitute for tuning the other knobs).
Everything else identical to Exp 1.


In [4]:
def rf_pipeline(train_frame, max_features='sqrt', min_samples_leaf=1, n_estimators=200):
    numeric_columns = train_frame.columns.difference(CATEGORICAL).tolist()
    preprocessor = ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), numeric_columns),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), CATEGORICAL),
    ])
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=n_estimators, min_samples_leaf=min_samples_leaf,
            max_features=max_features, n_jobs=-1, random_state=RANDOM_STATE)),
    ])

rf4 = rf_pipeline(X_train, n_estimators=400, min_samples_leaf=5)
rf4.fit(X_train, y_train)
auc4 = roc_auc_score(y_val, rf4.predict_proba(X_val)[:, 1])
print(f'Exp 4 (RF 400 trees, leaf=5) val AUC: {auc4:.5f}  delta vs Exp 1: {auc4 - LOG_REFERENCE["random_forest"]:+.5f}')


Exp 4 (RF 400 trees, leaf=5) val AUC: 0.95551  delta vs Exp 1: +0.00200


## Exp 5: random forest, second knob - max_features

Default 'sqrt' uses ~5 of 25+ features per split; with a few dominant features (Online boarding)
that can starve trees of the signal. `max_features=0.5` lets half the features compete per split.
One knob changed relative to Exp 4.


In [5]:
rf5 = rf_pipeline(X_train, n_estimators=400, min_samples_leaf=5, max_features=0.5)
rf5.fit(X_train, y_train)
auc5 = roc_auc_score(y_val, rf5.predict_proba(X_val)[:, 1])
print(f'Exp 5 (RF 400, leaf=5, max_feat=0.5) val AUC: {auc5:.5f}  delta vs Exp 4: {auc5 - auc4:+.5f}')
if auc5 >= auc4:
    best_rf_auc, best_rf_name, best_rf_kwargs = auc5, 'Exp 5', dict(n_estimators=400, min_samples_leaf=5, max_features=0.5)
    p_rf_val = rf5.predict_proba(X_val)[:, 1]
else:
    best_rf_auc, best_rf_name, best_rf_kwargs = auc4, 'Exp 4', dict(n_estimators=400, min_samples_leaf=5)
    p_rf_val = rf4.predict_proba(X_val)[:, 1]
print(f'Best RF this round: {best_rf_name} ({best_rf_auc:.5f})')


Exp 5 (RF 400, leaf=5, max_feat=0.5) val AUC: 0.95513  delta vs Exp 4: -0.00038


Best RF this round: Exp 4 (0.95551)


## Exp 6: the engineered feature set on ALL three model families

Same expanded features for logistic regression, the best RF config, and LightGBM - one content change,
tested per family so the log shows where features help and where they don't.
(LightGBM handles NaN natively, so `arr_delay_missing` is redundant for it; kept for a like-for-like set.)


In [6]:
# 6a: logistic regression + engineered features (its Exp 0 pipeline, extended set)
fe_numeric = X_train_fe.columns.difference(CATEGORICAL).tolist()
lr_pre = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), fe_numeric),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), CATEGORICAL),
])
lr6 = Pipeline([('preprocessor', lr_pre), ('classifier', LogisticRegression(max_iter=200))])
lr6.fit(X_train_fe, y_train)
auc6_lr = roc_auc_score(y_val, lr6.predict_proba(X_val_fe)[:, 1])
print(f'Exp 6a (logistic + FE) val AUC: {auc6_lr:.5f}  delta vs Exp 0: {auc6_lr - LOG_REFERENCE["logistic"]:+.5f}')


Exp 6a (logistic + FE) val AUC: 0.92796  delta vs Exp 0: +0.00750


In [7]:
# 6b: best RF config + engineered features
rf6 = rf_pipeline(X_train_fe, **best_rf_kwargs)
rf6.fit(X_train_fe, y_train)
auc6_rf = roc_auc_score(y_val, rf6.predict_proba(X_val_fe)[:, 1])
print(f'Exp 6b ({best_rf_name} config + FE) val AUC: {auc6_rf:.5f}  delta vs raw {best_rf_name}: {auc6_rf - best_rf_auc:+.5f}')
if auc6_rf > best_rf_auc:
    p_rf_val = rf6.predict_proba(X_val_fe)[:, 1]
    rf_source = f'Exp 6b ({best_rf_name} config + FE)'
else:
    rf_source = best_rf_name


Exp 6b (Exp 4 config + FE) val AUC: 0.95534  delta vs raw Exp 4: -0.00016


In [8]:
# 6c: LightGBM + engineered features (native categoricals, native NaN)
lgbm6 = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, num_leaves=63,
                           n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
lgbm6.fit(to_lgbm(X_train_fe), y_train, eval_set=[(to_lgbm(X_val_fe), y_val)], eval_metric='auc',
          callbacks=[lgb.early_stopping(50, verbose=False)])
auc6_lgbm = roc_auc_score(y_val, lgbm6.predict_proba(to_lgbm(X_val_fe))[:, 1])
print(f'Exp 6c (LightGBM + FE) val AUC: {auc6_lgbm:.5f}  delta vs Exp 2: {auc6_lgbm - LOG_REFERENCE["lightgbm"]:+.5f}  (best_iteration={lgbm6.best_iteration_})')
if auc6_lgbm >= LOG_REFERENCE['lightgbm']:
    p_lgbm_val = lgbm6.predict_proba(to_lgbm(X_val_fe))[:, 1]
    lgbm_source = 'Exp 6c (LightGBM + FE)'
    best_lgbm_trees = lgbm6.best_iteration_
    lgbm_uses_fe = True
else:
    lgbm2 = lgb.LGBMClassifier(n_estimators=418, learning_rate=0.05, num_leaves=63,
                               n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
    lgbm2.fit(to_lgbm(X_train), y_train)
    p_lgbm_val = lgbm2.predict_proba(to_lgbm(X_val))[:, 1]
    lgbm_source = 'Exp 2 config (raw, 418 trees)'
    best_lgbm_trees = 418
    lgbm_uses_fe = False


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Exp 6c (LightGBM + FE) val AUC: 0.95740  delta vs Exp 2: -0.00020  (best_iteration=363)


## Exp 7: blend the best RF with the best LightGBM

Different model families make different errors, so a plain average of their probabilities often beats either one.
Cheap to test, and the val split tells us honestly whether it helps here.


In [9]:
p_blend = 0.5 * p_rf_val + 0.5 * p_lgbm_val
auc7 = roc_auc_score(y_val, p_blend)
best_single_auc = max(best_rf_auc, auc6_rf, auc6_lgbm, LOG_REFERENCE['lightgbm'])
print(f'Blend: {rf_source} + {lgbm_source}')
print(f'Exp 7 (blend 50/50) val AUC: {auc7:.5f}')
print(f'vs best single model: {auc7 - best_single_auc:+.5f}')


Blend: Exp 4 + Exp 2 config (raw, 418 trees)
Exp 7 (blend 50/50) val AUC: 0.95690
vs best single model: -0.00070


## Round summary

Collects every number for the log. A candidate only earns a refit + submission file if it beats the
STANDING INCUMBENT (Exp 2 raw LightGBM, 0.95760), not just the other candidates in this round;
submitting stays your call from your own account. Checkpoint BEFORE more tuning: re-score the top
configs with 5-fold OOF - this holdout has now guided several choices and its estimates drift optimistic.


In [10]:
rows = [
    ('Exp 4  RF 400/leaf5', auc4),
    ('Exp 5  RF +max_feat0.5', auc5),
    ('Exp 6a LR + FE', auc6_lr),
    ('Exp 6b RF best + FE', auc6_rf),
    ('Exp 6c LGBM + FE', auc6_lgbm),
    ('Exp 7  blend', auc7),
]
summary = pd.DataFrame(rows, columns=['experiment', 'val_auc']).sort_values('val_auc', ascending=False)
summary['delta_vs_lgbm_raw'] = summary['val_auc'] - LOG_REFERENCE['lightgbm']
print(summary.to_string(index=False))
print(f'\n0.96 local target reached: {summary["val_auc"].max() >= 0.96}')


            experiment  val_auc  delta_vs_lgbm_raw
      Exp 6c LGBM + FE 0.957403          -0.000197
          Exp 7  blend 0.956900          -0.000700
   Exp 4  RF 400/leaf5 0.955508          -0.002092
   Exp 6b RF best + FE 0.955344          -0.002256
Exp 5  RF +max_feat0.5 0.955130          -0.002470
        Exp 6a LR + FE 0.927958          -0.029642

0.96 local target reached: False


In [ ]:
# CORRECTED ON REVIEW (2026-10-17) - this cell was NOT re-executed after the fix.
# As executed, it compared candidates only within this round, so it refit Exp 6c (0.95740)
# and wrote submission_round2.csv even though the standing incumbent, Exp 2 raw LightGBM
# (0.95760), scored higher. Do NOT submit that file. The guard below compares against the
# standing incumbent first; the round-3 notebook runs this corrected logic from the start.

STANDING_INCUMBENT = 0.95760  # Exp 2 raw LightGBM (experiments.md)
round_best_auc = max(auc7, auc6_lgbm, auc6_rf, best_rf_auc)

if round_best_auc <= STANDING_INCUMBENT:
    print(f'No round-2 candidate beat the standing incumbent (Exp 2 raw LightGBM, {STANDING_INCUMBENT:.5f}).')
    print('No new submission file written: submission_lgbm.csv (Exp 2) remains the best candidate.')
else:
    features_fe = add_features(features)
    test_fe = add_features(test_features)

    if auc7 > best_single_auc:
        # refit both blend members on full data
        if rf_source.startswith('Exp 6b'):
            rf_final = rf_pipeline(features_fe, **best_rf_kwargs)
            rf_final.fit(features_fe, target)
            p_rf_test = rf_final.predict_proba(test_fe)[:, 1]
        else:
            rf_final = rf_pipeline(features, **best_rf_kwargs)
            rf_final.fit(features, target)
            p_rf_test = rf_final.predict_proba(test_features)[:, 1]
        lgbm_final = lgb.LGBMClassifier(n_estimators=best_lgbm_trees, learning_rate=0.05, num_leaves=63,
                                        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
        if lgbm_uses_fe:
            lgbm_final.fit(to_lgbm(features_fe), target)
            p_lgbm_test = lgbm_final.predict_proba(to_lgbm(test_fe))[:, 1]
        else:
            lgbm_final.fit(to_lgbm(features), target)
            p_lgbm_test = lgbm_final.predict_proba(to_lgbm(test_features))[:, 1]
        test_probabilities = 0.5 * p_rf_test + 0.5 * p_lgbm_test
        winner = f'blend ({rf_source} + {lgbm_source})'
    elif auc6_lgbm >= max(best_rf_auc, auc6_rf):
        lgbm_final = lgb.LGBMClassifier(n_estimators=lgbm6.best_iteration_, learning_rate=0.05, num_leaves=63,
                                        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
        lgbm_final.fit(to_lgbm(features_fe), target)
        test_probabilities = lgbm_final.predict_proba(to_lgbm(test_fe))[:, 1]
        winner = 'Exp 6c (LightGBM + FE)'
    elif auc6_rf >= best_rf_auc:
        rf_final = rf_pipeline(features_fe, **best_rf_kwargs)
        rf_final.fit(features_fe, target)
        test_probabilities = rf_final.predict_proba(test_fe)[:, 1]
        winner = f'Exp 6b ({best_rf_name} config + FE)'
    else:
        rf_final = rf_pipeline(features, **best_rf_kwargs)
        rf_final.fit(features, target)
        test_probabilities = rf_final.predict_proba(test_features)[:, 1]
        winner = best_rf_name

    submission = pd.read_csv(data_dir / 'sample_submission.csv')
    submission['satisfaction'] = test_probabilities
    assert len(submission) == len(test)
    assert submission['id'].equals(test['id'])
    submission_path = competition_dir / 'submission_round2.csv'
    submission.to_csv(submission_path, index=False)
    print(f'Winner: {winner}')
    print(f'Saved {len(submission):,} predictions to {submission_path}.')



## Post-run correction (added on review, 2026-10-17)

The refit cell as executed compared candidates only within round 2, so it wrote
`submission_round2.csv` from Exp 6c (0.95740) - BELOW the standing incumbent, Exp 2 raw LightGBM
(0.95760). **Do not submit `submission_round2.csv`**; `submission_lgbm.csv` (Exp 2) remains the best
submission candidate. The refit cell above carries the corrected incumbent guard (marked, not
re-executed). Honest bottom line for round 2: no candidate beat the incumbent and the 0.96 local
target was not reached. Next step is the 5-fold OOF checkpoint, which decides what is real before
any more tuning.
